## Importaciones y configuración

In [1]:
# 01. IMPORTACIONES Y CONFIGURACIÓN PRINCIPAL

# Librerías generales
import os
import io
import random
import tempfile
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# Audio
import librosa
from IPython.display import Audio as IPythonAudio, display

# Datasets de Hugging Face
from datasets import load_from_disk, DatasetDict, Audio

# Transformers / Whisper
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Métricas
import evaluate

# Visualización
import matplotlib.pyplot as plt

In [2]:
# CONFIGURACIÓN PRINCIPAL DEL CUADERNO 07

# Semilla para reproducibilidad
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Modelo base que se va a utilizar en este entrenamiento

MODELO_BASE = "openai/whisper-small"

# Rutas principales

RUTA_DATASET_ORIGINAL = Path(r"C:\Lara\datasets\lara_whisper_dataset")
RUTA_MODELO_SALIDA = Path(r"C:\lara\modelos_entrenados\whisper_small_original")

RUTA_RESULTADOS = Path(r"C:\lara\resultados\07_whisper_small_original")
RUTA_LOGS = RUTA_RESULTADOS / "logs"
RUTA_GRAFICOS = RUTA_RESULTADOS / "graficos"
RUTA_METRICAS = RUTA_RESULTADOS / "metricas"

# Crear carpetas si no existen
for ruta in [
    RUTA_MODELO_SALIDA,
    RUTA_RESULTADOS,
    RUTA_LOGS,
    RUTA_GRAFICOS,
    RUTA_METRICAS
]:
    ruta.mkdir(parents=True, exist_ok=True)

# Configuración de idioma y tarea para Whisper

IDIOMA = "Spanish"
TAREA = "transcribe"

# Configuración de entrenamiento

NUM_TRAIN_EPOCHS = 5
LEARNING_RATE = 1e-5

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

# Batch efectivo: el tamaño de batch que realmente se utiliza para actualizar los pesos, teniendo en cuenta la acumulación de gradientes.
BATCH_EFECTIVO = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

# Configuración de evaluación y guardado

EVAL_STRATEGY = "epoch"
SAVE_STRATEGY = "epoch"

METRIC_FOR_BEST_MODEL = "wer"
GREATER_IS_BETTER = False

# Configuración de audio

SAMPLING_RATE = 16000

# Dispositivo de ejecución

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Configuración cargada correctamente")
print("----------------------------------------")
print("Modelo base:", MODELO_BASE)
print("Dataset original:", RUTA_DATASET_ORIGINAL)
print("Ruta salida modelo:", RUTA_MODELO_SALIDA)
print("Ruta resultados:", RUTA_RESULTADOS)
print("Dispositivo:", DEVICE)


print("----------------------------------------")
print("Épocas:", NUM_TRAIN_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch entrenamiento:", TRAIN_BATCH_SIZE)
print("Gradient accumulation steps:", GRADIENT_ACCUMULATION_STEPS)
print("Batch efectivo:", BATCH_EFECTIVO)

Configuración cargada correctamente
----------------------------------------
Modelo base: openai/whisper-small
Dataset original: C:\Lara\datasets\lara_whisper_dataset
Ruta salida modelo: C:\lara\modelos_entrenados\whisper_small_original
Ruta resultados: C:\lara\resultados\07_whisper_small_original
Dispositivo: cuda
----------------------------------------
Épocas: 5
Learning rate: 1e-05
Batch entrenamiento: 2
Gradient accumulation steps: 4
Batch efectivo: 8


## Carga del dataset original

In [3]:
# CARGA DEL DATASET ORIGINAL

# Cargamos el dataset original desde disco
dataset_original = load_from_disk(RUTA_DATASET_ORIGINAL)

# Evitamos la decodificación automática del audio
dataset_original = dataset_original.cast_column("audio", Audio(decode=False))

print("Dataset original cargado correctamente")
print("----------------------------------------")
print(dataset_original)
print("----------------------------------------")
print("Columnas:", dataset_original.column_names)
print("Número de registros:", len(dataset_original))

Dataset original cargado correctamente
----------------------------------------
Dataset({
    features: ['audio', 'texto'],
    num_rows: 40842
})
----------------------------------------
Columnas: ['audio', 'texto']
Número de registros: 40842


In [4]:
muestra = dataset_original[0]

print("Muestra cargada correctamente")
print("----------------------------------------")
print("Texto:", muestra["texto"])
print("Tipo campo audio:", type(muestra["audio"]))
print("Claves del campo audio:", muestra["audio"].keys())
print("Path audio:", muestra["audio"]["path"])
print("Tiene bytes:", muestra["audio"]["bytes"] is not None)

Muestra cargada correctamente
----------------------------------------
Texto: La bata tiene veinte botones.
Tipo campo audio: <class 'dict'>
Claves del campo audio: dict_keys(['bytes', 'path'])
Path audio: 6424622bdda7c982b10c7d39_1680106156.wav
Tiene bytes: True


In [5]:
# DIVISIÓN DEL DATASET ORIGINAL EN TRAIN, TEST Y EVAL

# Primera división: 80% entrenamiento y 20% temporal
split_inicial = dataset_original.train_test_split(
    test_size=0.20,
    seed=SEED
)

dataset_train = split_inicial["train"]
dataset_temporal = split_inicial["test"]

# Segunda división: del 20% temporal sacamos 80% test y 20% eval
split_temporal = dataset_temporal.train_test_split(
    test_size=0.20,
    seed=SEED
)

dataset_test = split_temporal["train"]
dataset_eval = split_temporal["test"]

# Agrupamos los splits en un DatasetDict
dataset_splits = DatasetDict({
    "train": dataset_train,
    "test": dataset_test,
    "eval": dataset_eval
})

print("División del dataset realizada correctamente")
print("----------------------------------------")
print(dataset_splits)
print("----------------------------------------")
print("Train:", len(dataset_splits["train"]))
print("Test:", len(dataset_splits["test"]))
print("Eval:", len(dataset_splits["eval"]))
print("----------------------------------------")
print("Total:", len(dataset_splits["train"]) + len(dataset_splits["test"]) + len(dataset_splits["eval"]))

División del dataset realizada correctamente
----------------------------------------
DatasetDict({
    train: Dataset({
        features: ['audio', 'texto'],
        num_rows: 32673
    })
    test: Dataset({
        features: ['audio', 'texto'],
        num_rows: 6535
    })
    eval: Dataset({
        features: ['audio', 'texto'],
        num_rows: 1634
    })
})
----------------------------------------
Train: 32673
Test: 6535
Eval: 1634
----------------------------------------
Total: 40842


In [6]:
# COMPROBACIÓN DE MUESTRAS POR SPLIT

for nombre_split in ["train", "test", "eval"]:
    muestra = dataset_splits[nombre_split][0]

    print("Split:", nombre_split)
    print("Texto:", muestra["texto"])
    print("Path audio:", muestra["audio"]["path"])
    print("Tiene bytes:", muestra["audio"]["bytes"] is not None)
    print("----------------------------------------")

Split: train
Texto: EL MAGO CONSIGUIÓ QUE EL ÁGUILA LLEGARÁ A LA LAGUNA.
Path audio: 6564622239048d0f405c669c_1716799914.wav
Tiene bytes: True
----------------------------------------
Split: test
Texto: EN LA LATA HAY LIMONADA.
Path audio: 662819171ef22d020bf25236_1744267180.wav
Tiene bytes: True
----------------------------------------
Split: eval
Texto: YO EMPUJO UN PAQUETE.
Path audio: 6564622239048d0f405c669c_1717406129.wav
Tiene bytes: True
----------------------------------------


## Carga del processor y del modelo Whisper Small

In [7]:
# CARGA DEL PROCESSOR Y DEL MODELO WHISPER SMALL

# Cargamos el processor de Whisper
processor = WhisperProcessor.from_pretrained(
    MODELO_BASE,
    language=IDIOMA,
    task=TAREA
)

# Cargamos el modelo base
modelo = WhisperForConditionalGeneration.from_pretrained(MODELO_BASE)

# Movemos el modelo al dispositivo disponible
modelo.to(DEVICE)

# Forzamos la generación en español y en modo transcripción
modelo.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=IDIOMA,
    task=TAREA
)

# Evitamos que Whisper use tokens suprimidos por defecto durante la generación
modelo.config.suppress_tokens = []

print("Processor y modelo cargados correctamente")
print("----------------------------------------")
print("Modelo base:", MODELO_BASE)
print("Idioma:", IDIOMA)
print("Tarea:", TAREA)
print("Dispositivo:", DEVICE)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Processor y modelo cargados correctamente
----------------------------------------
Modelo base: openai/whisper-small
Idioma: Spanish
Tarea: transcribe
Dispositivo: cuda


In [8]:
# PREPARACIÓN DEL DATASET PARA WHISPER

def preparar_dataset(muestra):
    # Recuperamos los bytes del audio guardados dentro del dataset
    audio_bytes = muestra["audio"]["bytes"]

    # Guardamos el audio temporalmente para que librosa pueda leerlo desde disco
    with tempfile.NamedTemporaryFile(suffix=".audio", delete=False) as archivo_temp:
        archivo_temp.write(audio_bytes)
        ruta_audio_temp = archivo_temp.name

    try:
        audio_array, sampling_rate = librosa.load(
            ruta_audio_temp,
            sr=SAMPLING_RATE,
            mono=True
        )

        # Extraemos las características de entrada para Whisper
        input_features = processor.feature_extractor(
            audio_array,
            sampling_rate=SAMPLING_RATE
        ).input_features[0]

        # Tokenizamos el texto objetivo
        labels = processor.tokenizer(
            muestra["texto"]
        ).input_ids

        return {
            "input_features": input_features,
            "labels": labels
        }

    finally:
        # Eliminamos el archivo temporal aunque haya error
        if os.path.exists(ruta_audio_temp):
            os.remove(ruta_audio_temp)


print("Función de preparación creada correctamente")

Función de preparación creada correctamente


In [9]:
# PRUEBA DE PREPARACIÓN CON UNA MUESTRA

muestra_preparada = preparar_dataset(dataset_splits["train"][0])

print("Muestra preparada correctamente")
print("----------------------------------------")
print("Tipo input_features:", type(muestra_preparada["input_features"]))
print("Longitud input_features:", len(muestra_preparada["input_features"]))
print("Tipo labels:", type(muestra_preparada["labels"]))
print("Número de tokens labels:", len(muestra_preparada["labels"]))
print("----------------------------------------")
print("Tokens:", muestra_preparada["labels"])

C:\Users\abelg\AppData\Local\Temp\ipykernel_9168\3599850597.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(
c:\Users\abelg\AppData\Local\Programs\Python\Python313\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Muestra preparada correctamente
----------------------------------------
Tipo input_features: <class 'numpy.ndarray'>
Longitud input_features: 80
Tipo labels: <class 'list'>
Número de tokens labels: 31
----------------------------------------
Tokens: [50258, 50262, 50359, 50363, 3158, 12191, 11601, 16596, 50, 10489, 46324, 42672, 46026, 14426, 24205, 32298, 4620, 32, 441, 2634, 38, 1899, 32172, 316, 9855, 9855, 38, 3979, 32, 13, 50257]


In [10]:
# APLICACIÓN DEL PREPROCESADO AL DATASET COMPLETO

dataset_preparado = DatasetDict()

for nombre_split in ["train", "test", "eval"]:
    print(f"Preparando split: {nombre_split}")
    print("----------------------------------------")

    dataset_preparado[nombre_split] = dataset_splits[nombre_split].map(
        preparar_dataset,
        remove_columns=dataset_splits[nombre_split].column_names,
        load_from_cache_file=False
    )

    print(f"Split {nombre_split} preparado correctamente")
    print("Registros:", len(dataset_preparado[nombre_split]))
    print("----------------------------------------")

print("Preprocesado completo finalizado")
print("----------------------------------------")
print(dataset_preparado)

Preparando split: train
----------------------------------------


Map:   0%|          | 0/32673 [00:00<?, ? examples/s]

C:\Users\abelg\AppData\Local\Temp\ipykernel_9168\3599850597.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(
c:\Users\abelg\AppData\Local\Programs\Python\Python313\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Split train preparado correctamente
Registros: 32673
----------------------------------------
Preparando split: test
----------------------------------------


Map:   0%|          | 0/6535 [00:00<?, ? examples/s]

Split test preparado correctamente
Registros: 6535
----------------------------------------
Preparando split: eval
----------------------------------------


Map:   0%|          | 0/1634 [00:00<?, ? examples/s]

Split eval preparado correctamente
Registros: 1634
----------------------------------------
Preprocesado completo finalizado
----------------------------------------
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 32673
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 6535
    })
    eval: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 1634
    })
})


In [11]:
# COMPROBACIÓN DEL DATASET PREPARADO

for nombre_split in ["train", "test", "eval"]:
    muestra = dataset_preparado[nombre_split][0]

    print("Split:", nombre_split)
    print("Columnas:", dataset_preparado[nombre_split].column_names)
    print("Tipo input_features:", type(muestra["input_features"]))
    print("Longitud input_features:", len(muestra["input_features"]))
    print("Tipo labels:", type(muestra["labels"]))
    print("Número de tokens labels:", len(muestra["labels"]))
    print("----------------------------------------")

Split: train
Columnas: ['input_features', 'labels']
Tipo input_features: <class 'list'>
Longitud input_features: 80
Tipo labels: <class 'list'>
Número de tokens labels: 31
----------------------------------------
Split: test
Columnas: ['input_features', 'labels']
Tipo input_features: <class 'list'>
Longitud input_features: 80
Tipo labels: <class 'list'>
Número de tokens labels: 16
----------------------------------------
Split: eval
Columnas: ['input_features', 'labels']
Tipo input_features: <class 'list'>
Longitud input_features: 80
Tipo labels: <class 'list'>
Número de tokens labels: 15
----------------------------------------


In [12]:
# GUARDADO DEL DATASET PREPARADO

RUTA_DATASET_PREPARADO_SMALL = Path(r"C:\Lara\datasets\lara_whisper_dataset_preparado_small_original")

dataset_preparado.save_to_disk(RUTA_DATASET_PREPARADO_SMALL)

print("Dataset preparado guardado correctamente")
print("----------------------------------------")
print("Ruta:", RUTA_DATASET_PREPARADO_SMALL)
print(dataset_preparado)

Saving the dataset (0/63 shards):   0%|          | 0/32673 [00:00<?, ? examples/s]

Saving the dataset (0/13 shards):   0%|          | 0/6535 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/1634 [00:00<?, ? examples/s]

Dataset preparado guardado correctamente
----------------------------------------
Ruta: C:\Lara\datasets\lara_whisper_dataset_preparado_small_original
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 32673
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 6535
    })
    eval: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 1634
    })
})


In [13]:
# CARGA DEL DATASET PREPARADO

# dataset_preparado = load_from_disk(RUTA_DATASET_PREPARADO_SMALL)

# print("Dataset preparado cargado correctamente")
# print("----------------------------------------")
# print(dataset_preparado)

## Data collator para Whisper

In [14]:
# DATA COLLATOR PARA WHISPER


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        # Separamos las características de audio
        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # Separamos las etiquetas tokenizadas
        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        # Sustituimos el padding por -100 para ignorarlo en la pérdida
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        # Si todas las etiquetas empiezan con el token BOS, lo eliminamos
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

print("Data collator creado correctamente")

Data collator creado correctamente


In [15]:
# PRUEBA DEL DATA COLLATOR

muestras_prueba = [
    dataset_preparado["train"][0],
    dataset_preparado["train"][1]
]

batch_prueba = data_collator(muestras_prueba)

print("Batch de prueba creado correctamente")
print("----------------------------------------")
print("Claves del batch:", batch_prueba.keys())
print("Shape input_features:", batch_prueba["input_features"].shape)
print("Shape labels:", batch_prueba["labels"].shape)
print("----------------------------------------")
print("Tipo input_features:", batch_prueba["input_features"].dtype)
print("Tipo labels:", batch_prueba["labels"].dtype)

Batch de prueba creado correctamente
----------------------------------------
Claves del batch: KeysView({'input_features': tensor([[[-0.6843, -0.3802, -0.3004,  ..., -0.8098, -0.8098, -0.8098],
         [-0.6815, -0.4066, -0.2496,  ..., -0.8098, -0.8098, -0.8098],
         [-0.5291, -0.5254, -0.1545,  ..., -0.8098, -0.8098, -0.8098],
         ...,
         [-0.8098, -0.8098, -0.7079,  ..., -0.8098, -0.8098, -0.8098],
         [-0.8098, -0.8098, -0.6282,  ..., -0.8098, -0.8098, -0.8098],
         [-0.8098, -0.8098, -0.8098,  ..., -0.8098, -0.8098, -0.8098]],

        [[-0.6285, -0.6285, -0.6285,  ..., -0.6285, -0.6285, -0.6285],
         [-0.6285, -0.6285, -0.6285,  ..., -0.6285, -0.6285, -0.6285],
         [-0.6285, -0.6285, -0.6285,  ..., -0.6285, -0.6285, -0.6285],
         ...,
         [-0.6285, -0.6285, -0.6285,  ..., -0.6285, -0.6285, -0.6285],
         [-0.6285, -0.6285, -0.6285,  ..., -0.6285, -0.6285, -0.6285],
         [-0.6285, -0.6285, -0.6285,  ..., -0.6285, -0.6285, -0.6

## Métricas de evaluación

In [16]:
# 10. MÉTRICAS DE EVALUACIÓN

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print("Métricas cargadas correctamente")
print("----------------------------------------")
print("WER:", wer_metric)
print("CER:", cer_metric)

Métricas cargadas correctamente
----------------------------------------
WER: EvaluationModule(name: "wer", module_type: "metric", features: {'predictions': Value(dtype='string', id='sequence'), 'references': Value(dtype='string', id='sequence')}, usage: """
Compute WER score of transcribed segments against references.

Args:
    references: List of references for each speech input.
    predictions: List of transcriptions to score.
    concatenate_texts (bool, default=False): Whether to concatenate all input texts or compute WER iteratively.

Returns:
    (float): the word error rate

Examples:

    >>> predictions = ["this is the prediction", "there is an other sample"]
    >>> references = ["this is the reference", "there is another one"]
    >>> wer = evaluate.load("wer")
    >>> wer_score = wer.compute(predictions=predictions, references=references)
    >>> print(wer_score)
    0.5
""", stored examples: 0)
CER: EvaluationModule(name: "cer", module_type: "metric", features: {'predic

In [17]:
# FUNCIÓN DE CÁLCULO DE MÉTRICAS

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Restauramos los tokens ignorados para poder decodificar las etiquetas
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decodificamos predicciones y referencias
    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    # Calculamos métricas principales
    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }


print("Función de métricas creada correctamente")

Función de métricas creada correctamente


## Configuración del entrenamiento

In [20]:
# CÁLCULO DE PASOS DE ENTRENAMIENTO Y WARMUP

steps_por_epoca = len(dataset_preparado["train"]) // BATCH_EFECTIVO
total_training_steps = steps_por_epoca * NUM_TRAIN_EPOCHS
warmup_steps = int(total_training_steps * 0.1)

print("Cálculo de pasos realizado correctamente")
print("----------------------------------------")
print("Registros train:", len(dataset_preparado["train"]))
print("Batch efectivo:", BATCH_EFECTIVO)
print("Steps por época:", steps_por_epoca)
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

Cálculo de pasos realizado correctamente
----------------------------------------
Registros train: 32673
Batch efectivo: 8
Steps por época: 4084
Total training steps: 20420
Warmup steps: 2042


In [22]:
# CONFIGURACIÓN DEL ENTRENAMIENTO

training_args = Seq2SeqTrainingArguments(
    output_dir=str(RUTA_MODELO_SALIDA),

    # Configuración principal del entrenamiento
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=warmup_steps,

    # Evaluación y guardado
    eval_strategy=EVAL_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=GREATER_IS_BETTER,

    # Generación durante evaluación
    predict_with_generate=True,
    generation_max_length=225,

    # Logging
    logging_dir=str(RUTA_LOGS),
    logging_strategy="steps",
    logging_steps=100,

    # Optimización
    fp16=torch.cuda.is_available(),

    # Evita intentar enviar datos a servicios externos
    report_to="none",

    # Columnas
    remove_unused_columns=False
)

print("Argumentos de entrenamiento configurados correctamente")
print("----------------------------------------")
print("Directorio de salida:", training_args.output_dir)
print("Épocas:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Batch train:", training_args.per_device_train_batch_size)
print("Batch eval:", training_args.per_device_eval_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Warmup steps:", training_args.warmup_steps)
print("Evaluación:", training_args.eval_strategy)
print("Guardado:", training_args.save_strategy)
print("Mejor modelo según:", training_args.metric_for_best_model)
print("FP16:", training_args.fp16)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Argumentos de entrenamiento configurados correctamente
----------------------------------------
Directorio de salida: C:\lara\modelos_entrenados\whisper_small_original
Épocas: 5
Learning rate: 1e-05
Batch train: 2
Batch eval: 2
Gradient accumulation: 4
Warmup steps: 2042
Evaluación: IntervalStrategy.EPOCH
Guardado: SaveStrategy.EPOCH
Mejor modelo según: wer
FP16: True


## Creación del Trainer

In [23]:
# CREACIÓN DEL TRAINER

trainer = Seq2SeqTrainer(
    model=modelo,
    args=training_args,
    train_dataset=dataset_preparado["train"],
    eval_dataset=dataset_preparado["eval"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor
)

print("Trainer creado correctamente")
print("----------------------------------------")
print("Modelo:", MODELO_BASE)
print("Dataset train:", len(dataset_preparado["train"]))
print("Dataset eval:", len(dataset_preparado["eval"]))
print("Directorio de salida:", RUTA_MODELO_SALIDA)

Trainer creado correctamente
----------------------------------------
Modelo: openai/whisper-small
Dataset train: 32673
Dataset eval: 1634
Directorio de salida: C:\lara\modelos_entrenados\whisper_small_original


In [24]:
# COMPROBACIÓN PREVIA DEL TRAINER

print("Comprobación del trainer")
print("----------------------------------------")
print("Modelo en dispositivo:", next(modelo.parameters()).device)
print("FP16 activado:", training_args.fp16)
print("Batch efectivo:", BATCH_EFECTIVO)
print("Warmup steps:", warmup_steps)
print("Evaluación cada:", training_args.eval_strategy)
print("Guardado cada:", training_args.save_strategy)

Comprobación del trainer
----------------------------------------
Modelo en dispositivo: cuda:0
FP16 activado: True
Batch efectivo: 8
Warmup steps: 2042
Evaluación cada: IntervalStrategy.EPOCH
Guardado cada: SaveStrategy.EPOCH


## Entrenamiento del modelo

In [25]:
resultado_entrenamiento = trainer.train()

Epoch,Training Loss,Validation Loss,Wer,Cer
1,0.886696,0.218967,1.544033,1.481406
2,0.363024,0.109106,0.583073,0.585496
3,0.132195,0.083558,0.654494,0.637352
4,0.041792,0.077626,0.469384,0.381053
5,0.007648,0.074616,0.537073,0.446380


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


In [26]:
# GUARDADO DEL MEJOR MODELO ENTRENADO

trainer.save_model(str(RUTA_MODELO_SALIDA))
processor.save_pretrained(str(RUTA_MODELO_SALIDA))

print("Mejor modelo guardado correctamente")
print("----------------------------------------")
print("Ruta:", RUTA_MODELO_SALIDA)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Mejor modelo guardado correctamente
----------------------------------------
Ruta: C:\lara\modelos_entrenados\whisper_small_original


In [27]:
# GUARDADO DE MÉTRICAS DEL ENTRENAMIENTO

df_metricas_entrenamiento = pd.DataFrame(trainer.state.log_history)

ruta_metricas_entrenamiento = RUTA_METRICAS / "metricas_entrenamiento_whisper_small_original.csv"

df_metricas_entrenamiento.to_csv(
    ruta_metricas_entrenamiento,
    index=False,
    encoding="utf-8-sig"
)

print("Métricas de entrenamiento guardadas correctamente")
print("----------------------------------------")
print("Ruta:", ruta_metricas_entrenamiento)

df_metricas_entrenamiento.tail()

Métricas de entrenamiento guardadas correctamente
----------------------------------------
Ruta: C:\lara\resultados\07_whisper_small_original\metricas\metricas_entrenamiento_whisper_small_original.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_wer,eval_cer,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
205,0.010408,0.684170,1.294674e-07,4.945094,20200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206,0.006454,0.540041,7.506936e-08,4.969578,20300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
207,0.007648,1.241098,2.067127e-08,4.994063,20400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
208,NaN,NaN,NaN,5.000000,20425,0.074616,0.537073,0.44638,2280.9161,0.716,0.358,NaN,NaN,NaN,NaN,NaN
209,NaN,NaN,NaN,5.000000,20425,NaN,NaN,NaN,NaN,NaN,NaN,89267.3951,1.83,0.229,4.714475e+19,0.951757


In [28]:
# EVALUACIÓN FINAL SOBRE EL CONJUNTO DE TEST

metricas_test = trainer.evaluate(
    eval_dataset=dataset_preparado["test"],
    metric_key_prefix="test"
)

print("Evaluación sobre test finalizada")
print("----------------------------------------")

for clave, valor in metricas_test.items():
    print(f"{clave}: {valor}")

Training Loss,Validation Loss,Epoch,Wer,Cer
0.007648,0.082088,5,0.442387,0.372565


Evaluación sobre test finalizada
----------------------------------------
test_loss: 0.08208800107240677
test_wer: 0.44238688325796427
test_cer: 0.372564834935426


In [29]:
# RESUMEN DE RESULTADOS DEL MODELO WHISPER SMALL

resumen_resultados = {
    "modelo": MODELO_BASE,
    "dataset": "original",
    "train_registros": len(dataset_preparado["train"]),
    "test_registros": len(dataset_preparado["test"]),
    "eval_registros": len(dataset_preparado["eval"]),
    "test_loss": metricas_test["test_loss"],
    "test_wer": metricas_test["test_wer"],
    "test_cer": metricas_test["test_cer"],
    "epocas": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_train": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "batch_efectivo": BATCH_EFECTIVO,
    "warmup_steps": warmup_steps
}

df_resumen_resultados = pd.DataFrame([resumen_resultados])

ruta_resumen_resultados = RUTA_METRICAS / "resumen_resultados_whisper_small_original.csv"

df_resumen_resultados.to_csv(
    ruta_resumen_resultados,
    index=False,
    encoding="utf-8-sig"
)

print("Resumen de resultados guardado correctamente")
print("----------------------------------------")
print("Ruta:", ruta_resumen_resultados)

df_resumen_resultados

Resumen de resultados guardado correctamente
----------------------------------------
Ruta: C:\lara\resultados\07_whisper_small_original\metricas\resumen_resultados_whisper_small_original.csv


,modelo,dataset,train_registros,test_registros,eval_registros,test_loss,test_wer,test_cer,epocas,learning_rate,batch_train,gradient_accumulation_steps,batch_efectivo,warmup_steps
0,openai/whisper-small,original,32673,6535,1634,0.082088,0.442387,0.372565,5,0.00001,2,4,8,2042


El modelo whisper-small entrenado sobre el dataset original obtiene mejores resultados que el primer modelo whisper-base, pero la mejora no es tan grande como cabría esperar.

La evolución de las métricas muestra cierto sobreajuste, especialmente al comparar la pérdida de entrenamiento con las métricas de validación y test.

Esto refuerza la idea de que la calidad del dataset es un factor limitante importante. El siguiente paso lógico sería entrenar whisper-small sobre una versión más limpia del dataset.